# Familiar customers vs new customers

## The business decision

A retailer wants to estimate **next-month spending for newly acquired customers** so it can decide how much onboarding support to offer. Historical data contains several visits for each existing customer.

The question is not simply, *Which test produces the better score?* It is:

> **Which test matches the customers we will face when the model is used?**

Estimated time: 8–10 minutes. No file submission is required. Keep one sentence explaining your decision.

## 1. Notice the repeated customers

Each row is a visit—not a different customer. The same customer can therefore appear several times.

| customer | visit | activity | next-month spending |
|---|---:|---:|---:|
| CUST-001 | 1 | low | $92 |
| CUST-001 | 2 | high | $111 |
| CUST-002 | 1 | medium | $74 |
| CUST-002 | 2 | high | $86 |

**Pair prompt:** If CUST-001 has one visit in training and another in validation, is that evidence about a *new* customer?

## 2. Predict before running

Two tests are available:

- **Test A — Familiar customers:** visits are split randomly, so some customers appear in both training and validation.
- **Test B — New customers:** all visits for a customer stay together, so validation contains customers never seen in training.

Commit to an answer: **Which test will report the larger average error—and which test matches the business decision?**

In [ ]:
#@title 3. Run the comparison { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
rng = np.random.default_rng(7)
n_customers, rows_per_customer = 160, 4
customer_id = np.repeat([f'CUST-{i:03d}' for i in range(n_customers)], rows_per_customer)
customer_value = np.repeat(rng.normal(100, 28, n_customers), rows_per_customer)
visit_activity = rng.normal(0, 1, n_customers * rows_per_customer)
next_month_spending = customer_value + 6 * visit_activity + rng.normal(0, 4, len(customer_id))
data = pd.DataFrame({'customer_id': customer_id, 'visit_activity': visit_activity, 'next_month_spending': next_month_spending})
features, target = ['customer_id', 'visit_activity'], 'next_month_spending'
model = Pipeline([('prepare', ColumnTransformer([('customer', OneHotEncoder(handle_unknown='ignore'), ['customer_id']), ('numeric', StandardScaler(), ['visit_activity'])])), ('model', Ridge(alpha=0.2))])
familiar_train, familiar_valid = train_test_split(data, test_size=0.25, random_state=42)
familiar_overlap = len(set(familiar_train.customer_id) & set(familiar_valid.customer_id))
model.fit(familiar_train[features], familiar_train[target])
familiar_mae = mean_absolute_error(familiar_valid[target], model.predict(familiar_valid[features]))
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(data, groups=data.customer_id))
new_train, new_valid = data.iloc[train_idx], data.iloc[valid_idx]
new_overlap = len(set(new_train.customer_id) & set(new_valid.customer_id))
model.fit(new_train[features], new_train[target])
new_mae = mean_absolute_error(new_valid[target], model.predict(new_valid[features]))
results = pd.DataFrame({'Test': ['A — Familiar customers', 'B — New customers'], 'Customers seen on both sides': [familiar_overlap, new_overlap], 'Average error (MAE)': [familiar_mae, new_mae]})
display(results.style.format({'Average error (MAE)': '${:,.2f}'}).hide(axis='index'))
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(['A: familiar customers', 'B: new customers'], [familiar_mae, new_mae], color=['#7aa6c2', '#e07a5f'])
ax.set_ylabel('Average prediction error ($)')
ax.set_title('The test that matches new customers is harder—and more credible')
ax.bar_label(bars, labels=[f'${familiar_mae:,.2f}', f'${new_mae:,.2f}'], padding=3)
ax.spines[['top', 'right']].set_visible(False)
plt.ylim(0, max(familiar_mae, new_mae) * 1.25)
plt.show()
print(f'\nTest A reused {familiar_overlap} customers across training and validation.')
print(f'Test B reused {new_overlap} customers across training and validation.')

## 4. Make the business decision

Complete this sentence and compare it with a peer:

> **Test ___ is the credible test for newly acquired customers because ___.**

### Debrief

Test A looks impressive partly because the model has already encountered many of the same customer identities. Test B is harder because it evaluates genuinely unseen customers—the population named in the business decision.

This is a **validation-design mismatch**, not future-information leakage. Leakage is a separate problem: it occurs when a feature contains information that would not exist at the moment the prediction is made.